
1. Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan,

Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.

Gunakan model dengan kernel linier.

Gunakan model dengan kernel polynomial.

Gunakan model dengan kernel RBF.

Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

In [35]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Membaca dataset
df = pd.read_csv('voice.csv')

# Memisahkan fitur dan label
X = df.drop('label', axis=1)
y = df['label']

# Menyimpan hasil
hasil = []

# Percobaan split 70:30 dan 80:20
for test_size, split in [(0.30, '70:30'), (0.20, '80:20')]:

    # Membagi data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    # Normalisasi data
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Kernel Linear
    model_linear = SVC(kernel='linear')
    model_linear.fit(X_train, y_train)

    pred_linear = model_linear.predict(X_test)
    acc_linear = accuracy_score(y_test, pred_linear)

    hasil.append([split, 'Linear', acc_linear])

    # Kernel Polynomial
    model_poly = SVC(kernel='poly')
    model_poly.fit(X_train, y_train)

    pred_poly = model_poly.predict(X_test)
    acc_poly = accuracy_score(y_test, pred_poly)

    hasil.append([split, 'Polynomial', acc_poly])

    # Kernel RBF
    model_rbf = SVC(kernel='rbf')
    model_rbf.fit(X_train, y_train)

    pred_rbf = model_rbf.predict(X_test)
    acc_rbf = accuracy_score(y_test, pred_rbf)

    hasil.append([split, 'RBF', acc_rbf])

# Membuat tabel hasil
hasil_df = pd.DataFrame(
    hasil,
    columns=['Split Data', 'Kernel', 'Akurasi']
)

hasil_df['Akurasi'] = hasil_df['Akurasi'].round(4)

hasil_df

,Split Data,Kernel,Akurasi
0,70:30,Linear,0.9790
1,70:30,Polynomial,0.9600
2,70:30,RBF,0.9832
3,80:20,Linear,0.9748
4,80:20,Polynomial,0.9558
5,80:20,RBF,0.9826



2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import cv2
import os

# Image directories
train_img = "/content/drive/MyDrive/Images/training/"
test_img = "/content/drive/MyDrive/Images/test/"

# Memisahkan gambar dan label
images = []
labels = []

for label in os.listdir(train_img):
    folder = os.path.join(train_img, label)

    if os.path.isdir(folder):
        for filename in os.listdir(folder):
            path = os.path.join(folder, filename)

            img = cv2.imread(path)

            if img is not None:
                images.append(img)
                labels.append(label)

print('Jumlah data:', len(images))
print('Label:', set(labels))

Jumlah data: 240
Label: {'day', 'night'}


In [43]:
# Ekstraksi fitur histogram
features = []

for img in images:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()

    features.append(hist)

X = np.array(features)
y = np.array(labels)

print('Ukuran fitur:', X.shape)
print('Ukuran label:', y.shape)

Ukuran fitur: (240, 256)
Ukuran label: (240,)


In [44]:
# Membagi data training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Data training:', len(X_train))
print('Data testing :', len(X_test))

Data training: 192
Data testing : 48


In [45]:
# Membuat model SVM dengan kernel RBF
model = SVC(kernel='rbf')

# Training model
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Menghitung akurasi
accuracy = accuracy_score(y_test, y_pred)

print('Akurasi SVM RBF:', accuracy)

Akurasi SVM RBF: 1.0


In [46]:
# Percobaan beberapa nilai C dan gamma
C_values = [0.1, 1, 10, 100]
gamma_values = ['scale', 0.01, 0.1, 1]

hasil = []

for C in C_values:
    for gamma in gamma_values:
        model = SVC(
            kernel='rbf',
            C=C,
            gamma=gamma
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        hasil.append([C, gamma, accuracy])

# Membuat tabel hasil
hasil_df = pd.DataFrame(
    hasil,
    columns=['C', 'Gamma', 'Accuracy']
)

print(hasil_df)

        C  Gamma  Accuracy
0     0.1  scale  0.958333
1     0.1   0.01  0.770833
2     0.1    0.1  0.770833
3     0.1      1  0.854167
4     1.0  scale  1.000000
5     1.0   0.01  0.770833
6     1.0    0.1  0.833333
7     1.0      1  1.000000
8    10.0  scale  1.000000
9    10.0   0.01  0.854167
10   10.0    0.1  0.958333
11   10.0      1  1.000000
12  100.0  scale  1.000000
13  100.0   0.01  0.958333
14  100.0    0.1  1.000000
15  100.0      1  1.000000


In [47]:
best = hasil_df.loc[hasil_df['Accuracy'].idxmax()]

print('Parameter terbaik:')
print('C     :', best['C'])
print('Gamma :', best['Gamma'])
print('Akurasi:', best['Accuracy'])

Parameter terbaik:
C     : 1.0
Gamma : scale
Akurasi: 1.0


Berdasarkan percobaan klasifikasi gambar day dan night menggunakan fitur histogram dan SVM kernel RBF dengan pembagian data 80:20, diperoleh akurasi sebesar 100%. Setelah dilakukan tuning hyperparameter C dan gamma, kombinasi terbaik yang diperoleh adalah C = 1.0 dan gamma = scale dengan akurasi tetap sebesar 100%. Karena itu saya mengambil kesimpulan bahwa model SVM dengan kernel RBF dan parameter tersebut sudah mampu mengklasifikasikan gambar day dan night dengan sangat baik pada dataset yang digunakan.